## Start

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import datetime
import plotly.graph_objects as go
import hw2fintools as ft


load_dotenv()
ticker = ''
path_stockdata = os.path.join(os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList'), ticker)


today = datetime.date.today()
cutoff = today.year - 20
cutoff10 = today.year - 10
cutoff5 = today.year - 5
cutoff3 = today.year - 3

## Data Imports

In [ ]:
%%capture

df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Annual_10K-s1v1.csv'), index_col=0)
df0 = df0.dropna(subset=['FiscalYear'])
df0[['FiscalYear', 'FiscalMonth']] = df0[['FiscalYear', 'FiscalMonth']].astype(int)

In [ ]:
df0

In [ ]:
price0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Price_History-s1v1.csv'), index_col=0)
price0['Date'] = pd.to_datetime(price0['Date'])
price0 = price0[price0['Date'].dt.year >= cutoff]
fiscal_month = df0['FiscalMonth'].iloc[0]
price0['FiscalYear'] = price0['Date'].apply(lambda x: ft.to_fiscal_year(date=x, fiscal_end_month=fiscal_month))

price0aggr = price0.groupby(price0['FiscalYear']).agg(
    SharePriceMin=pd.NamedAgg(column='PricePerShare', aggfunc='min'),
    SharePriceMax=pd.NamedAgg(column='PricePerShare', aggfunc='max'),
    SharePriceMean=pd.NamedAgg(column='PricePerShare', aggfunc='mean'),
    SharePriceMedian=pd.NamedAgg(column='PricePerShare', aggfunc='median'),
)


In [ ]:
price0aggr

In [ ]:
%%capture

ps_df0 = df0[['FiscalYear', 'FiscalMonth', 'RevPS', 'EarnPS', 'OpCashPS', 'FreeCashPS', 'DivPS', 'HighPrice', 'LowPrice']]
ps_df1 = ps_df0.merge(price0aggr, on=['FiscalYear'])


In [ ]:
ps_df1

## Revenue Per Share

In [ ]:
rps_fig1 = go.Figure(data=[
   go.Bar(name='Rev', x=ps_df1['FiscalYear'].tail(20), y=ps_df1['RevPS'].tail(20), offsetgroup=1, marker_color='blue'),
])
rps_fig1.update_xaxes(dtick=1)
rps_fig1.update_layout(yaxis_title='RPS', xaxis_title='FiscalYear', title='Year20 RevPerShare', template='plotly_dark')
rps_fig1.show()

In [ ]:
rps_fig2 = go.Figure(data=[
    go.Bar(name='MeanPR', x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMean'].tail(20)/ ps_df1['RevPS'].tail(20), 1), offsetgroup=1, marker_color='blue'),
    go.Bar(name="MedianPR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMedian'].tail(20)/ ps_df1['RevPS'].tail(20), 1), offsetgroup=2, marker_color='purple'),
    go.Bar(name="LowPR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMin'].tail(20)/ ps_df1['RevPS'].tail(20), 1), offsetgroup=3, marker_color='green'),
    go.Bar(name="HighPR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMax'].tail(20)/ ps_df1['RevPS'].tail(20), 1), offsetgroup=4, marker_color='red'),


])
rps_fig2.update_xaxes(dtick=1)
rps_fig2.update_layout(barmode='group')
rps_fig2.update_layout(yaxis_title='Ratio', xaxis_title='FiscalYear', title='RPS', template='plotly_dark')
rps_fig2.show()

## Earning Per Share

In [ ]:
eps_fig1 = go.Figure(data=[
   go.Bar(name='Earnings', x=ps_df1['FiscalYear'].tail(20), y=ps_df1['EarnPS'].tail(20), offsetgroup=1, marker_color='blue'),
])
eps_fig1.update_xaxes(dtick=1)
eps_fig1.update_layout(yaxis_title='EPS', xaxis_title='FiscalYear', title='EPS', template='plotly_dark')
eps_fig1.show()

In [ ]:
eps_fig2 = go.Figure(data=[
    go.Bar(name='MeanER', x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMean'].tail(20)/ ps_df1['EarnPS'].tail(20), 1), offsetgroup=1, marker_color='blue'),
    go.Bar(name="MedianER", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMedian'].tail(20)/ ps_df1['EarnPS'].tail(20), 1), offsetgroup=2, marker_color='purple'),
    go.Bar(name="LowER", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMin'].tail(20)/ ps_df1['EarnPS'].tail(20), 1), offsetgroup=3, marker_color='green'),
    go.Bar(name="HighER", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMax'].tail(20)/ ps_df1['EarnPS'].tail(20), 1), offsetgroup=4, marker_color='red'),


])
eps_fig2.update_xaxes(dtick=1)
eps_fig2.update_layout(barmode='group')
eps_fig2.update_layout(yaxis_title='Ratio', xaxis_title='FiscalYear', title='Price to Earnings', template='plotly_dark')
eps_fig2.show()

## Operating Cash

In [ ]:
ocps_fig1 = go.Figure(data=[
   go.Bar(name='OpCash', x=ps_df1['FiscalYear'].tail(20), y=ps_df1['OpCashPS'].tail(20), offsetgroup=1, marker_color='blue'),
])
ocps_fig1.update_xaxes(dtick=1)
ocps_fig1.update_layout(yaxis_title='OCPS', xaxis_title='FiscalYear', title='OCPS', template='plotly_dark')
ocps_fig1.show()

In [ ]:
ocps_fig2 = go.Figure(data=[
    go.Bar(name='MeanOCR', x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMean'].tail(20)/ ps_df1['OpCashPS'].tail(20), 1), offsetgroup=1, marker_color='blue'),
    go.Bar(name="MedianOCR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMedian'].tail(20)/ ps_df1['OpCashPS'].tail(20), 1), offsetgroup=2, marker_color='purple'),
    go.Bar(name="LowOCR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMin'].tail(20)/ ps_df1['OpCashPS'].tail(20), 1), offsetgroup=3, marker_color='green'),
    go.Bar(name="HighOCR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMax'].tail(20)/ ps_df1['OpCashPS'].tail(20), 1), offsetgroup=4, marker_color='red'),


])
ocps_fig2.update_xaxes(dtick=1)
ocps_fig2.update_layout(barmode='group')
ocps_fig2.update_layout(yaxis_title='Ratio', xaxis_title='FiscalYear', title='OCPS', template='plotly_dark')
ocps_fig2.show()

## Free Cash

In [ ]:
fcps_fig1 = go.Figure(data=[
   go.Bar(name='FreeCash', x=ps_df1['FiscalYear'].tail(20), y=ps_df1['FreeCashPS'].tail(20), offsetgroup=1, marker_color='blue'),
])
fcps_fig1.update_xaxes(dtick=1)
fcps_fig1.update_layout(yaxis_title='FCPS', xaxis_title='FiscalYear', title='FCPS', template='plotly_dark')
fcps_fig1.show()

In [ ]:
fcps_fig2 = go.Figure(data=[
    go.Bar(name='MeanFCR', x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMean'].tail(20)/ ps_df1['FreeCashPS'].tail(20), 1), offsetgroup=1, marker_color='blue'),
    go.Bar(name="MedianFCR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMedian'].tail(20)/ ps_df1['FreeCashPS'].tail(20), 1), offsetgroup=2, marker_color='purple'),
    go.Bar(name="LowFCR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMin'].tail(20)/ ps_df1['FreeCashPS'].tail(20), 1), offsetgroup=3, marker_color='green'),
    go.Bar(name="HighFCR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMax'].tail(20)/ ps_df1['FreeCashPS'].tail(20), 1), offsetgroup=4, marker_color='red'),


])
fcps_fig2.update_xaxes(dtick=1)
fcps_fig2.update_layout(barmode='group')
fcps_fig2.update_layout(yaxis_title='Ratio', xaxis_title='FiscalYear', title='FCPS', template='plotly_dark')
fcps_fig2.show()

## Dividend

In [ ]:
dps_fig1 = go.Figure(data=[
   go.Bar(name='DivPS', x=ps_df1['FiscalYear'].tail(20), y=ps_df1['DivPS'].tail(20), offsetgroup=1, marker_color='blue'),
])
dps_fig1.update_xaxes(dtick=1)
dps_fig1.update_layout(yaxis_title='DPS', xaxis_title='FiscalYear', title='DPS', template='plotly_dark')
dps_fig1.show()

In [ ]:
dps_fig2 = go.Figure(data=[
    go.Bar(name='MeanDR', x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMean'].tail(20)/ ps_df1['DivPS'].tail(20), 1), offsetgroup=1, marker_color='blue'),
    go.Bar(name="MedianDR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMedian'].tail(20)/ ps_df1['DivPS'].tail(20), 1), offsetgroup=2, marker_color='purple'),
    go.Bar(name="LowDR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMin'].tail(20)/ ps_df1['DivPS'].tail(20), 1), offsetgroup=3, marker_color='green'),
    go.Bar(name="HighDR", x=ps_df1['FiscalYear'].tail(20), y=round(ps_df1['SharePriceMax'].tail(20)/ ps_df1['DivPS'].tail(20), 1), offsetgroup=4, marker_color='red'),


])
dps_fig2.update_xaxes(dtick=1)
dps_fig2.update_layout(barmode='group')
dps_fig2.update_layout(yaxis_title='Ratio', xaxis_title='FiscalYear', title='DPS', template='plotly_dark')
dps_fig2.show()